# GradCAM

This notebook implements **GradCAM1D**, an adaptation of Grad-CAM for 1D convolutional networks, to visualize which regions of the ECG signal drive the model's predictions for each diagnostic class (NORM, MI, STTC, CD, HYP).

## Implementation

A `GradCAM1D` class is implemented as a context manager that registers forward and backward hooks on a target convolutional layer (the last `conv2` of the final residual block) to capture its activations and gradients during a forward/backward pass. For a given input segment and target class, the class logit is backpropagated, the gradients are global-average-pooled across the temporal dimension to obtain per-channel importance weights, and a weighted sum of the activation maps is computed and passed through ReLU to keep only positive contributions. The resulting 1D heatmap is normalized to the range [0, 1].

Since ECG records are split into overlapping windowed segments rather than processed as a whole, segment-level heatmaps are stitched back together into a single patient-level heatmap (`compute_stitched_heatmap`). Each segment's heatmap is interpolated to the window length, accumulated into a full-length array using its stride offset, and overlapping regions are averaged by dividing by the overlap count before a final normalization.

Patient-level class probabilities are computed via max-pooling across all segment predictions belonging to the same record (`predict_patient_max_pool`), consistent with the record-level aggregation strategy used elsewhere in the evaluation pipeline.

## Visualization

The `plot_stitched_results` function overlays the reconstructed heatmap on top of the raw ECG trace for a selectable subset of leads (here leads 0, 3, 7, and 11), using a shared color scale to represent relative gradient importance, with dotted vertical lines marking the boundaries between stitched segment windows.

## Experiment

For patient 159, GradCAM was run for both the **Baseline** model (4 residual blocks, BCE loss) and the **Final** model (2 residual blocks, the architecture selected after the ablation study), across all five diagnostic classes. This allows a direct visual comparison of which ECG regions each model configuration attends to when producing its predictions, and how that attention shifts between the baseline and the optimized final model.

### Importing libraries


In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import traceback
from typing import Dict, Tuple, List

- Defining activation functions

In [2]:
ACTIVATIONS = {
    "relu": nn.ReLU,
    "gelu": nn.GELU,
    "mish": nn.Mish,
}

In [3]:
def make_activation(act):
    if isinstance(act, str):
        if act not in ACTIVATIONS:
            raise ValueError(f"unknown activation '{act}'. options: {list(ACTIVATIONS)}")
        return ACTIVATIONS[act]() #returns the PyTorch activation layer
    return act()

## Deep Neural Network Architecture


In [4]:
class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=7, stride=1, dropout=0.2, activation="relu"):
        super().__init__()
        pad = kernel // 2
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel, stride=stride, padding=pad, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel, padding=pad, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.drop  = nn.Dropout(dropout)
        self.act1  = make_activation(activation)
        self.act2  = make_activation(activation)

        """
        Skip connection: add the original input back into the output
        """
        self.shortcut = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm1d(out_ch)
        ) if (stride != 1 or in_ch != out_ch) else nn.Identity()

    def forward(self, x):
        out = self.act1(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        out = self.act2(out + self.shortcut(x))
        return out

In [5]:
class ECGResNet(nn.Module):
    """
    1D ResNet for multi-lead ECG classification.

    Input  : (batch, n_leads, signal_len)   e.g. (B, 12, 500)
    Output : (batch, n_classes)             raw logits

    Depth is set by `channels`: each entry is one residual block.
        [64, 128, 256, 512] -> 4 blocks (original)
        [64, 128]           -> 2 blocks
    The first block runs at stride 1 (the stem already downsamples);
    every subsequent block uses stride 2.
    """
    def __init__(self, n_leads=12, n_classes=5, dropout=0.2, activation="relu",
                 channels=(64, 128), stem_ch=64):
        super().__init__()
        channels = list(channels)
        
        self.stem = nn.Sequential(
            nn.Conv1d(n_leads, stem_ch, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(stem_ch),
            make_activation(activation),
            nn.MaxPool1d(3, stride=2, padding=1)
        )

        blocks = []
        in_ch = stem_ch
        for i, out_ch in enumerate(channels):
            stride = 1 if i == 0 else 2
            blocks.append(
                ResBlock1D(in_ch, out_ch, stride=stride,
                           dropout=dropout, activation=activation)
            )
            in_ch = out_ch
        self.blocks = nn.Sequential(*blocks)

        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(channels[-1], n_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.pool(x).squeeze(-1)
        return self.head(x)

## Loading pretrained models

In [6]:
def load_pretrained_resnet(checkpoint_path: str, channels: Tuple[int, ...], device="cpu") -> nn.Module:
    model = ECGResNet(n_leads=12, n_classes=5, channels=channels, activation="relu")
    checkpoint = torch.load(checkpoint_path, map_location=torch.device(device), weights_only=False)

    state_dict = checkpoint['model_state'] if isinstance(checkpoint, dict) and 'model_state' in checkpoint else checkpoint
    model.load_state_dict(state_dict)

    #Anti GradCam runtime graph errors
    for module in model.modules():
        if isinstance(module, (nn.ReLU, nn.GELU, nn.Mish)) and hasattr(module, 'inplace'):
            module.inplace = False

    model = model.to(device)
    model.eval()
    return model

## GradCAM1D Realization

In [7]:
class GradCAM1D:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.forward_hook = None
        self.backward_hook = None
        
    def __enter__(self):
        self.forward_hook = self.target_layer.register_forward_hook(self._save_activation)
        self.backward_hook = self.target_layer.register_full_backward_hook(self._save_gradient)
        return self
        
    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.forward_hook: self.forward_hook.remove()
        if self.backward_hook: self.backward_hook.remove()

    def _save_activation(self, module, input, output):
        self.activations = output.detach()
        
    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
        
    def generate_heatmap(self, input_tensor: torch.Tensor, class_idx: int) -> Tuple[np.ndarray, np.ndarray]:
        self.model.eval()
        
        with torch.set_grad_enabled(True):
            input_tensor = input_tensor.requires_grad_(True)
            output = self.model(input_tensor) 
            loss = output[0, class_idx]
            
            self.model.zero_grad()
            loss.backward()
        
        weights = torch.mean(self.gradients, dim=-1)[0]
        activation = self.activations[0]
        
        heatmap = torch.sum(weights.unsqueeze(-1) * activation, dim=0)
        heatmap = torch.clamp(heatmap, min=0)
        
        heatmap_max = heatmap.max()
        if heatmap_max > 0:
            heatmap = heatmap / heatmap_max
            
        return heatmap.cpu().numpy(), output.detach().cpu().numpy()

## Utility functions

In [8]:
def get_patient_segments(ids_test: np.ndarray, target_patient_id: int):
    """Finds all corresponding window indices for a given explicit patient ID."""
    segment_indices = np.where(ids_test == target_patient_id)[0]
    
    if len(segment_indices) == 0:
        raise ValueError(f"Patient ID {target_patient_id} not found in the test set.")
        
    return target_patient_id, segment_indices

In [9]:
def predict_patient_max_pool(model: nn.Module, X_test: np.ndarray, segment_indices: np.ndarray, device: str) -> Tuple[int, float]:
    """Computes patient-level evaluations using a global max-pooling method."""
    all_probs = []
    with torch.no_grad():
        for seg_idx in segment_indices:
            seg_sample = torch.from_numpy(X_test[seg_idx]).float().unsqueeze(0).to(device)
            probs = torch.sigmoid(model(seg_sample))
            all_probs.append(probs)
            
    stacked_probs = torch.cat(all_probs, dim=0)
    max_probs, _ = torch.max(stacked_probs, dim=0)
    #predicted_class = torch.argmax(max_probs, dim=-1).item()
    return max_probs.cpu().numpy()

In [10]:
def compute_stitched_heatmap(
    model: nn.Module, target_layer: nn.Module, X_test: np.ndarray, segment_indices: np.ndarray,
    predicted_class: int, window_length: int, stride: int, full_length: int, device: str
) -> Tuple[np.ndarray, np.ndarray]:
    full_heatmap = np.zeros(full_length)
    overlap_count = np.zeros(full_length)
    full_signal = np.zeros((12, full_length))

    with GradCAM1D(model, target_layer) as cam_extractor:
        for step, seg_idx in enumerate(segment_indices):
            seg_sample = torch.from_numpy(X_test[seg_idx]).float().unsqueeze(0).to(device)
            heatmap, _ = cam_extractor.generate_heatmap(seg_sample, class_idx=predicted_class)
            
            # Interpolate low-resolution heatmap up to current sub-window scale
            heatmap_resized = np.interp(
                np.linspace(0, len(heatmap) - 1, window_length), 
                np.arange(len(heatmap)), 
                heatmap
            )
            
            start = step * stride
            end = start + window_length
            
            full_heatmap[start:end] += heatmap_resized
            overlap_count[start:end] += 1
            full_signal[:, start:end] = X_test[seg_idx]

    # Normalize across overlaps
    full_heatmap /= np.where(overlap_count > 0, overlap_count, 1)
    if full_heatmap.max() > 0:
        full_heatmap /= full_heatmap.max()
        
    return full_heatmap, full_signal

In [11]:
def plot_stitched_results(
    full_signal: np.ndarray, full_heatmap: np.ndarray, patient_id: int, 
    model_name: str, class_label: str, max_prob: float, stride: int, leads_to_plot:List[int], save_path: str):

    num_leads = len(leads_to_plot)
    fig, axes = plt.subplots(num_leads, 1, figsize=(14, 2.2 * num_leads), sharex=True)
    if num_leads == 1:
        axes = [axes]


    time_ticks = np.arange(full_signal.shape[-1])
    
    for i, lead_idx in enumerate(leads_to_plot):
        ax = axes[i]
        target_signal = full_signal[lead_idx, :]
        y_min, y_max = target_signal.min(), target_signal.max()
        
        # 1. Plot the ECG trace for this specific channel
        ax.plot(time_ticks, target_signal, color='black', linewidth=1.2, label=f'Lead {lead_idx}', zorder=2)
        
        # 2. Overlay the SAME temporal heatmap onto this channel
        mesh = ax.imshow(
            full_heatmap[None, :], cmap='jet', alpha=0.30, aspect='auto',
            extent=[time_ticks.min(), time_ticks.max(), y_min, y_max],
            origin='lower', zorder=1, vmin=0.0, vmax=1.0
        )
        
        # Subplot styling
        ax.set_ylabel("Amplitude", fontsize=9)
        ax.set_ylim(y_min - 0.2, y_max + 0.2)
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(True, linestyle='--', alpha=0.4)
        
        # Draw the overlapping window boundary anchors
        for boundary in range(stride, len(time_ticks), stride):
            ax.axvline(x=boundary, color='gray', linestyle=':', alpha=0.4, zorder=0)

    axes[-1].set_xlabel("Time Step (Samples)", fontsize=11)
    
    
    fig.suptitle(f"Multi-Lead Stitched GradCAM - {model_name} | Patient: {patient_id} | Class: {class_label} ({max_prob:.4f})", 
                fontsize=14, y=0.99)
    
    # Adjust layout to make room for a unified colorbar on the right side
    fig.subplots_adjust(right=0.88, hspace=0.15)
    cbar_ax = fig.add_axes([0.91, 0.15, 0.015, 0.7])
    fig.colorbar(mesh, cax=cbar_ax, label='Relative Class Gradient Importance (0.0 - 1.0)')
    
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

## Execution

In [ ]:
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Execution Configuration Matrix
    MODEL_CONFIGS = [
        {"name": "Baseline", "path": "/best_resnet_depth4.pt", "channels": (64, 128, 256, 512)},
        {"name": "Final", "path": "/best_resnet_act_relu.pt", "channels": (64, 128)}
    ]
    CLASS_NAMES = {0: "NORM", 1: "MI", 2: "STTC", 3: "CD", 4: "HYP"}
    WINDOW_LENGTH, STRIDE = 300, 150
    TARGET_PATIENT_ID = 159
    
    output_path = "/grad_cam_plots_fixed/"
    LEAD_TO_PLOT = [0,3,7,11] 

    # Load Source Tensors
    X_test_data = np.load("/X_test_seg.npy")
    ids_test_data = np.load("/ids_test.npy")

    if X_test_data.shape[1] != 12:
        X_test_data = np.transpose(X_test_data, (0, 2, 1))

    # Identify targeted metrics
    patient_id, segment_idxs = get_patient_segments(ids_test_data, TARGET_PATIENT_ID)
    print(f"Targeting Patient ID: {patient_id} | Found {len(segment_idxs)} segments at indices: {segment_idxs}")
    
    total_reconstructed_length = WINDOW_LENGTH + (len(segment_idxs) - 1) * STRIDE

    for config in MODEL_CONFIGS:
        print(f"\nProcessing configuration: [{config['name']}] Model")
        try:
            
            net = load_pretrained_resnet(config["path"], channels=config["channels"], device=device)
            target_conv = net.blocks[-1].conv2
            
            all_max_probs = predict_patient_max_pool(net, X_test_data, segment_idxs, device)
            
            for class_idx, class_str in CLASS_NAMES.items():
                class_prob = all_max_probs[class_idx]
                print(f"-> Analyzing Class: '{class_str}' | Prob score: {class_prob:.4f}")
                
                # Compute gradient activations for this specific class
                heatmap_vec, signal_mat = compute_stitched_heatmap(
                    net, target_conv, X_test_data, segment_idxs,
                    class_idx, WINDOW_LENGTH, STRIDE, total_reconstructed_length, device
                )
                
                # Visualize with the newly specified lead
                out_filename = f"{output_path}gradcam_patient_{patient_id}_{config['name'].lower()}_depth{len(config['channels'])}_class_{class_str}_lead_{LEAD_TO_PLOT}.png"
                plot_stitched_results(
                    signal_mat, heatmap_vec, patient_id, config['name'], 
                    class_str, class_prob, STRIDE, LEAD_TO_PLOT, out_filename
                )
                print(f"   Saved to: {out_filename}")
            
        except Exception as e:
            print(f"-> Fail state encountered processing {config['name']} Model:")
            traceback.print_exc()

Targeting Patient ID: 159 | Found 5 segments at indices: [795 796 797 798 799]

Processing configuration: [Baseline] Model
-> Analyzing Class: 'NORM' | Prob score: 0.4227
   Saved to: C:/Users/alege/Desktop/Medic/XAI_proj/grad_cam_plots_fixed/gradcam_patient_159_baseline_depth4_class_NORM_lead_[0, 3, 7, 11].png
-> Analyzing Class: 'MI' | Prob score: 0.3263
   Saved to: C:/Users/alege/Desktop/Medic/XAI_proj/grad_cam_plots_fixed/gradcam_patient_159_baseline_depth4_class_MI_lead_[0, 3, 7, 11].png
-> Analyzing Class: 'STTC' | Prob score: 0.0600
   Saved to: C:/Users/alege/Desktop/Medic/XAI_proj/grad_cam_plots_fixed/gradcam_patient_159_baseline_depth4_class_STTC_lead_[0, 3, 7, 11].png
-> Analyzing Class: 'CD' | Prob score: 0.6354
   Saved to: C:/Users/alege/Desktop/Medic/XAI_proj/grad_cam_plots_fixed/gradcam_patient_159_baseline_depth4_class_CD_lead_[0, 3, 7, 11].png
-> Analyzing Class: 'HYP' | Prob score: 0.0069
   Saved to: C:/Users/alege/Desktop/Medic/XAI_proj/grad_cam_plots_fixed/gradca